# Where does UMaze physics become readable?

This notebook reproduces and strengthens the repository's UMaze linear-probe methodology. It asks **which physical variables can be recovered with a simple linear map, from which internal tensor, and at which model layer**. The variables are 2-D position, 2-D velocity, 2-D acceleration, speed, heading, acceleration magnitude, and acceleration direction.

The key distinction is temporal context:

- **DINO is a per-frame visual encoder.** At layer $\ell$, it produces a token tensor for one image. We probe its CLS token, the mean of its patch tokens, and the projected aggregate. Position is decoded from one frame's feature $h_t^\ell$. Motion is decoded primarily from the change between frames: $\Delta h_t^\ell$ for velocity and $\Delta^2 h_t^\ell$ for acceleration.
- **The predictor is temporally contextual.** It receives visual tokens together with action/proprioception and causal history. We probe the mean predictor visual-token activation at each predictor block. Because this activation has already mixed temporal and action context, its per-slot feature may legitimately contain motion information.

The model weights are frozen: only a ridge-linear decoder is fitted. Therefore, **readable** means that the variable is arranged accessibly enough for a linear map to recover it on held-out data. It does **not** mean the variable is stored in a single neuron, uniquely represented, or causally used by the planner.

## One-click Colab setup

The default Colab path is deliberately credential-free. It clones the exact probe branch and downloads a versioned activation cache produced from the R0 checkpoint and 128 deterministic UMaze windows. The cache contains frozen intermediate activations, states, actions, and window identifiers; the remaining cells refit every probe and regenerate every graph.

This mode is the practical reproducible walkthrough: collaborators do not need the original 58 GB trajectory directory or an R2 key. To recompute activations from raw frames instead, set `UMAZE_FORCE_RECOMPUTE=1` and provide `UMAZE_CHECKPOINT` plus `UMAZE_DATA_DIR` before running this cell.

In [ ]:
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

REPO_URL = "https://github.com/danielhacobian/temporal-straightening.git"
REPO_BRANCH = "codex/umaze-physics-probes"
COLAB_REPO = Path("/content/temporal-straightening")
ASSET_URL = "https://github.com/danielhacobian/temporal-straightening/releases/download/umaze-probe-colab-v1/activation_cache_pooled.npz"
ASSET_SHA256 = "46e8863b2ef82ea79025a54e472b1357495999d224f4938fa541952df13ad598"

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

if IN_COLAB:
    if not (COLAB_REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--branch", REPO_BRANCH, "--single-branch",
            REPO_URL, str(COLAB_REPO),
        ], check=True)
    else:
        subprocess.run([
            "git", "-C", str(COLAB_REPO), "fetch", "origin", REPO_BRANCH,
        ], check=True)
        subprocess.run([
            "git", "-C", str(COLAB_REPO), "checkout", "--detach", "FETCH_HEAD",
        ], check=True)
    os.chdir(COLAB_REPO)

    force_recompute = os.environ.get("UMAZE_FORCE_RECOMPUTE", "0") == "1"
    if force_recompute:
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "decord", "einops", "omegaconf", "hydra-core",
        ], check=True)
    else:
        asset_dir = Path("/content/umaze_probe_assets")
        asset_dir.mkdir(parents=True, exist_ok=True)
        cache_path = asset_dir / "activation_cache_pooled.npz"
        if not cache_path.exists() or sha256(cache_path) != ASSET_SHA256:
            temporary = cache_path.with_suffix(".download")
            print(f"Downloading public probe cache to {cache_path} ...")
            urllib.request.urlretrieve(ASSET_URL, temporary)
            observed = sha256(temporary)
            if observed != ASSET_SHA256:
                temporary.unlink(missing_ok=True)
                raise RuntimeError(
                    f"Activation-cache checksum mismatch: {observed} != {ASSET_SHA256}"
                )
            temporary.replace(cache_path)
        os.environ.setdefault("UMAZE_ACTIVATION_CACHE", str(cache_path))

    os.environ.setdefault("UMAZE_PROBE_OUTPUT", "/content/umaze_probe_results")

print({
    "in_colab": IN_COLAB,
    "repo": str(Path.cwd()),
    "mode": "full_recompute" if os.environ.get("UMAZE_FORCE_RECOMPUTE") == "1" else "public_cache",
    "cache": os.environ.get("UMAZE_ACTIVATION_CACHE"),
})

## Experimental design

For a layer representation $h_t^\ell$:

$$p_t=(x_t,y_t),\qquad v_t=\\frac{p_{t+1}-p_t}{\Delta t},\qquad a_t=\\frac{v_{t+1}-v_t}{\Delta t}$$

Let a cached representation have shape `[window, time, feature] = [N,T,D]`. For four sampled frames, a frame probe sees $N\times T$ rows of $D$ features, a first-difference probe sees $N\times(T-1)$ rows, and a second-difference probe sees $N\times(T-2)$ rows. The probe never receives pixels, model parameters, future labels, or explicit XY coordinates (except in the separate position-only control).

Primary feature/target pairs:

| Family | Tensor probed | Position target | Velocity target | Acceleration target |
|---|---|---|---|---|
| DINO | CLS, mean patch tokens, or projected aggregate at block $\ell$ | $h_t^\ell \rightarrow (x_t,y_t)$ | $h_{t+1}^\ell-h_t^\ell \rightarrow (p_{t+1}-p_t)/\Delta t$ | $h_{t+2}^\ell-2h_{t+1}^\ell+h_t^\ell \rightarrow (v_{t+1}-v_t)/\Delta t$ |
| Predictor | mean contextual visual tokens at block $\ell$ | $h_t^\ell \rightarrow (x_t,y_t)$ | $h_t^\ell \rightarrow (v_{x,t},v_{y,t})$ | $h_t^\ell \rightarrow (a_{x,t},a_{y,t})$ |

Every probe standardizes features using **training-set statistics only** and fits ridge regression with $\lambda=10$. A single probe is fitted independently for each `(family, layer, representation kind, target, feature construction, split)` combination. We compare an episode-held-out split with a spatially blocked split and include shuffled-label, position-only, and position-residualized controls.

In [ ]:
from pathlib import Path
import csv, json, os, sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts.umaze_probe_walkthrough import (
    align_representation, bootstrap_metric_ci, build_motion_targets,
    direction_scores, episode_group_split, fit_probe, load_activation_cache,
    mask_slow_directions, readability_onset, regression_scores, residualize_against_position,
    save_activation_cache, shuffled_label_score, spatial_holdout_split,
)

plt.style.use("seaborn-v0_8-whitegrid")

def write_rows(path, rows):
    if not rows:
        return
    fieldnames = list(dict.fromkeys(key for row in rows for key in row))
    with Path(path).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader(); writer.writerows(rows)

def show_rows(rows, columns=None, limit=20):
    rows = list(rows)[:limit]
    if not rows:
        print("(no rows)"); return
    columns = columns or list(rows[0])
    widths = {
        key: min(42, max(len(key), *(len(f"{row.get(key, '')}") for row in rows)))
        for key in columns
    }
    print(" | ".join(key.ljust(widths[key]) for key in columns))
    print("-+-".join("-" * widths[key] for key in columns))
    for row in rows:
        print(" | ".join(f"{row.get(key, '')}"[:widths[key]].ljust(widths[key]) for key in columns))

def grouped(rows, keys):
    result = {}
    for row in rows:
        key = tuple(row[name] for name in keys)
        result.setdefault(key, []).append(row)
    return result

def configured_path(environment_key, *candidates):
    if os.environ.get(environment_key):
        return Path(os.environ[environment_key]).expanduser()
    for candidate in candidates:
        candidate = Path(candidate).expanduser()
        if candidate.exists():
            return candidate
    return Path(candidates[0]).expanduser()
SEED = 0
RIDGE = 10.0
MAX_WINDOWS = 512
NUM_FRAMES = 4
FRAME_SKIP = 5
STEP_DT = 1.0  # set to the environment seconds-per-step if physical units are needed
BATCH_SIZE = 8

WORKSPACE_ROOT = ROOT.resolve().parent
CHECKPOINT = configured_path(
    "UMAZE_CHECKPOINT",
    ROOT / "baseline_artifacts/checkpoints/umaze_q1_retrain/r0_direction_only/checkpoints/model_20.pth",
    WORKSPACE_ROOT / "reference/r0_direction_only/checkpoints/model_20.pth",
)
DATA_DIR = configured_path(
    "UMAZE_DATA_DIR",
    Path.home() / "data/point_maze",
    WORKSPACE_ROOT / "data/point_maze",
)
OUTPUT_DIR = Path(os.environ.get("UMAZE_PROBE_OUTPUT", ROOT / "baseline_artifacts/analysis/umaze_probe_walkthrough"))
CACHE = Path(os.environ.get("UMAZE_ACTIVATION_CACHE", OUTPUT_DIR / "activation_cache_pooled.npz"))
DEVICE = os.environ.get("UMAZE_DEVICE", "cuda:0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({"checkpoint": str(CHECKPOINT), "data": str(DATA_DIR), "cache": str(CACHE), "device": DEVICE})

## 1. Collect or load intermediate activations

The cache contains states, actions, sampled `(episode, start)` windows, and frozen intermediate tensors. Representation names have the form `family/layer/kind`:

- `dino/L/cls`: the global CLS token after DINO block `L`; shape `[N,T,D_dino]`.
- `dino/L/pooled_patches`: the arithmetic mean over all spatial patch tokens after block `L`; shape `[N,T,D_dino]`.
- `dino/L/projected_aggregate`: the repository's projected/aggregated DINO feature; shape `[N,T,D_proj]`.
- `predictor/L/pooled_visual`: the arithmetic mean over contextual predictor visual tokens after predictor block `L`; shape `[N,T,D_pred]`.

The probe sees one of these vectors, or its temporal difference, **not the entire network and not raw images**. Individual patch vectors are intentionally excluded from this lightweight Colab cache because retaining every patch at every layer can consume several gigabytes. Delete the cache to force re-extraction after changing the checkpoint, dataset, or window configuration.

In [ ]:
if CACHE.exists():
    representations, states, actions, choices, cache_metadata = load_activation_cache(CACHE)
    print(f"Loaded {len(representations)} representations from {CACHE}")
else:
    import torch
    from datasets.img_transforms import default_transform
    from datasets.point_maze_dset import PointMazeDataset
    from scripts.probe_umaze_layers import collect_activations, load_checkpoint, sample_windows

    if not CHECKPOINT.exists() or not DATA_DIR.exists():
        raise FileNotFoundError(
            "Set UMAZE_CHECKPOINT and UMAZE_DATA_DIR, or copy the checkpoint/data from R2, "
            "before running activation extraction."
        )
    device = torch.device(DEVICE)
    modules = load_checkpoint(CHECKPOINT, device)
    use_frame_files = (DATA_DIR / "obses" / "episode_000_frame_000.pth").exists()
    dataset = PointMazeDataset(
        data_path=str(DATA_DIR), transform=default_transform(224),
        normalize_action=True, use_frame_files=use_frame_files,
    )
    choices = sample_windows(dataset, MAX_WINDOWS, FRAME_SKIP, NUM_FRAMES, SEED)
    representations, states, actions = collect_activations(
        modules, dataset, choices, BATCH_SIZE, FRAME_SKIP, NUM_FRAMES, device,
        include_kinds={"cls", "pooled_patches", "projected_aggregate", "pooled_visual"},
    )
    cache_metadata = {
        "checkpoint": str(CHECKPOINT), "data_dir": str(DATA_DIR), "seed": SEED,
        "max_windows": MAX_WINDOWS, "num_frames": NUM_FRAMES,
        "frameskip": FRAME_SKIP, "ridge": RIDGE,
    }
    save_activation_cache(CACHE, representations, states, actions, choices, cache_metadata)
    print(f"Saved pooled activation cache to {CACHE}")

inventory = [
    {"representation": name, "shape": str(value.shape), "size_mb": value.nbytes / 2**20}
    for name, value in sorted(representations.items())
]
show_rows(inventory)
print("Total cached representation memory (MB):", sum(row["size_mb"] for row in inventory))

## 2. Construct physical targets and inspect shortcut risk

Position comes from `state[..., :2] = (x,y)`. Instantaneous velocity uses `state[..., 2:4] = (v_x,v_y)` when present. For a DINO first-difference feature spanning frames $t$ and $t+1$, the label is the matching displacement velocity $(p_{t+1}-p_t)/\Delta t$. Acceleration is the matching finite difference of velocity. With `FRAME_SKIP=5` and `STEP_DT=1`, adjacent cached frames are separated by $\Delta t=5$ environment steps; change `STEP_DT` if one simulator step corresponds to a known number of seconds.

Before fitting a model probe, inspect whether speed and acceleration already correlate with maze location. Strong spatial structure is exactly the shortcut a single-frame DINO probe could exploit.

In [ ]:
targets = build_motion_targets(states, FRAME_SKIP, STEP_DT)
position = targets["position"].reshape(-1, 2)
speed = targets["speed"].reshape(-1)
acceleration_magnitude = targets["acceleration_magnitude"].reshape(-1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
coverage = axes[0].scatter(position[:, 0], position[:, 1], c=speed, s=8, cmap="viridis")
axes[0].set(
    title="Where the cached trajectories move quickly or slowly",
    xlabel="Agent x position (environment units)",
    ylabel="Agent y position (environment units)", aspect="equal",
)
fig.colorbar(coverage, ax=axes[0], label="Speed (position units per environment step)")
axes[1].hist(speed, bins=40)
axes[1].set(
    title="Speed labels used by the probes",
    xlabel="Speed (position units per environment step)",
    ylabel="Number of frame-level samples",
)
axes[2].hist(acceleration_magnitude[np.isfinite(acceleration_magnitude)], bins=40)
axes[2].set(
    title="Acceleration-magnitude labels used by the probes",
    xlabel="Acceleration magnitude (position units per step²)",
    ylabel="Number of frame-level samples",
)
fig.savefig(OUTPUT_DIR / "dataset_motion_overview.png", dpi=180)
plt.show()

weighted, x_edges, y_edges = np.histogram2d(position[:, 0], position[:, 1], bins=12, weights=speed)
counts, _, _ = np.histogram2d(position[:, 0], position[:, 1], bins=[x_edges, y_edges])
speed_map = np.divide(weighted, counts, out=np.full_like(weighted, np.nan), where=counts > 0)
plt.figure(figsize=(8, 6)); plt.imshow(
    speed_map.T, origin="lower", cmap="viridis", aspect="equal",
    extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
)
plt.colorbar(label="Mean speed (position units per environment step)")
plt.xlabel("Agent x position (environment units)")
plt.ylabel("Agent y position (environment units)")
plt.title("Mean speed by spatial bin (shortcut diagnostic)")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "speed_by_position.png", dpi=180); plt.show()

## 3. Leakage-resistant evaluation splits

- **Episode-held-out:** complete trajectories are assigned to train or test. This prevents nearby or overlapping windows from the same episode crossing the boundary.
- **Spatial holdout:** the upper 20% of window-anchor Y positions is test-only, with a buffer band removed from training. Change the axis/direction to rotate through every arm and corner.

The split is made at the window level and then shared by every representation and layer.

In [ ]:
episode_train, episode_test = episode_group_split(choices, test_fraction=0.2, seed=SEED)
anchor_position = targets["position"][:, 0]
spatial_train, spatial_test, spatial_config = spatial_holdout_split(
    anchor_position, axis=1, quantile=0.8, high=True, buffer_fraction=0.05
)
splits = {
    "episode_holdout": (episode_train, episode_test),
    "spatial_holdout": (spatial_train, spatial_test),
}
print({name: (len(train), len(test)) for name, (train, test) in splits.items()})
print("Spatial split:", spatial_config)

plt.figure(figsize=(6, 5))
plt.scatter(anchor_position[spatial_train, 0], anchor_position[spatial_train, 1], s=18, label="Probe-training windows")
plt.scatter(anchor_position[spatial_test, 0], anchor_position[spatial_test, 1], s=18, label="Unseen-region test windows")
plt.legend(); plt.gca().set_aspect("equal"); plt.title("Spatial generalization test: held-out upper-Y region")
plt.xlabel("Window-start x position (environment units)")
plt.ylabel("Window-start y position (environment units)")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "spatial_split.png", dpi=180); plt.show()

## 4. Fit every layer with controls

For each representation, the code calls `align_representation` to construct the correct feature/label pair without mixing windows. It flattens only the time rows belonging to the selected training or test windows. `fit_probe` then standardizes the feature columns from the training set, fits a ridge-linear map, and applies that frozen map to held-out rows. No gradients pass into DINO or the predictor.

For each fitted probe we record:

- held-out $R^2$, RMSE, and a bootstrap interval;
- a shuffled-label null;
- a position-only baseline predicting the same target from XY;
- residual-motion readability after subtracting the component predictable from XY.

Interpret the controls together: a credible motion result should beat the shuffled-label 95th percentile, beat the position-only decoder, remain positive after removing the portion of the target predictable from XY, and survive the spatial holdout. The residual score is especially important for raw per-frame DINO motion probes.

In [ ]:
def evaluate_representation(name, rep, variable, mode, split_name, train_idx, test_idx):
    # Construct h_t, Δh_t, or Δ²h_t and align it with the matching physical label.
    features, labels, position_context = align_representation(rep, targets, variable, mode)
    # Fit only the linear readout; the cached world-model representation stays frozen.
    truth, prediction, _ = fit_probe(features, labels, train_idx, test_idx, RIDGE)
    scores = regression_scores(truth, prediction)
    ci = bootstrap_metric_ci(truth, prediction, "r2", repeats=300, seed=SEED)
    # Destroy feature/label pairing to estimate a finite-sample null distribution.
    shuffled = shuffled_label_score(
        features, labels, train_idx, test_idx, RIDGE, repeats=20, seed=SEED
    )
    # Test how much of the target can be predicted from maze XY alone.
    pos_truth, pos_prediction, _ = fit_probe(
        position_context, labels, train_idx, test_idx, RIDGE
    )
    # Remove the target component explained linearly by XY, then probe what remains.
    residual_labels = residualize_against_position(labels, position_context, train_idx, RIDGE)
    residual_truth, residual_prediction, _ = fit_probe(
        features, residual_labels, train_idx, test_idx, RIDGE
    )
    family, layer, kind = name.split("/", 2)
    return {
        "representation": name, "family": family, "layer": int(layer), "kind": kind,
        "variable": variable, "mode": mode, "split": split_name,
        **scores, "ci_low": ci[0], "ci_high": ci[1],
        "shuffled_q95": float(np.quantile(shuffled, 0.95)),
        "position_only_r2": regression_scores(pos_truth, pos_prediction)["r2"],
        "position_residual_r2": regression_scores(residual_truth, residual_prediction)["r2"],
    }

rows = []
for split_name, (train_idx, test_idx) in splits.items():
    for name, rep in sorted(representations.items()):
        family = name.split("/", 1)[0]
        specs = [("position", "frame")]
        if family == "dino":
            specs += [("velocity", "frame"), ("velocity", "delta")]
            if rep.shape[1] >= 3:
                specs += [("acceleration", "frame"), ("acceleration", "second_delta")]
        else:
            specs += [("velocity", "frame"), ("acceleration", "frame")]
        for variable, mode in specs:
            rows.append(evaluate_representation(
                name, rep, variable, mode, split_name, train_idx, test_idx
            ))

metrics = rows
write_rows(OUTPUT_DIR / "layerwise_cartesian_metrics.csv", metrics)
show_rows(metrics, limit=8)

## 5. Where each Cartesian variable becomes readable

The primary DINO curves use a raw frame for position, a first temporal difference for velocity, and a second temporal difference for acceleration. Predictor curves use contextual per-slot features. The horizontal axis is the transformer-block index; the vertical axis is held-out $R^2$. $R^2=1$ is perfect prediction, $R^2=0$ is no better than predicting the held-out mean, and negative values are worse than that baseline. Shaded regions are bootstrap intervals over held-out rows. Compare curves within a panel; do not treat raw $R^2$ values for different targets as equally difficult tasks.

In [ ]:
def primary_mode(family, variable):
    if family == "predictor" or variable == "position":
        return "frame"
    return {"velocity": "delta", "acceleration": "second_delta"}[variable]

def plot_layer_curves(frame, split="episode_holdout", filename="readability_by_layer.png"):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), constrained_layout=True)
    for ax, variable in zip(axes, ["position", "velocity", "acceleration"]):
        candidates = [row for row in frame if row["split"] == split and row["variable"] == variable]
        for (family, kind), group in grouped(candidates, ["family", "kind"]).items():
            group = sorted(
                [row for row in group if row["mode"] == primary_mode(family, variable)],
                key=lambda row: row["layer"],
            )
            if not group:
                continue
            label = f"{family}: {kind}"
            layer = np.asarray([row["layer"] for row in group])
            ax.plot(layer, [row["r2"] for row in group], marker="o", label=label)
            ax.fill_between(layer, [row["ci_low"] for row in group], [row["ci_high"] for row in group], alpha=0.12)
        ax.axhline(0, color="black", lw=1)
        ax.set(
            title=f"Linear readability of 2-D {variable}",
            xlabel="Transformer block index ℓ (0 = earliest cached block)",
            ylabel="Held-out coefficient of determination R²",
        )
        ax.legend(fontsize=7)
    fig.savefig(OUTPUT_DIR / filename, dpi=180)
    plt.show()

plot_layer_curves(metrics)
plot_layer_curves(metrics, split="spatial_holdout", filename="readability_by_layer_spatial_holdout.png")

## 6. Is DINO motion static or genuinely temporal?

This is the core shortcut check. `frame` asks a linear probe to infer motion from one frozen DINO frame feature, even though DINO itself has no temporal input. `delta` supplies $h_{t+1}-h_t$ for velocity, and `second_delta` supplies $h_{t+2}-2h_{t+1}+h_t$ for acceleration. A high raw-frame score that disappears on the spatial holdout can be explained by location. A temporal-difference score that beats the position-only baseline and survives in unseen regions is stronger evidence that **change in representation space tracks physical motion**.

In [ ]:
dino_motion = [
    row for row in metrics
    if row["family"] == "dino" and row["variable"] in ("velocity", "acceleration")
]
fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
for ax, split_name, variable in zip(
    axes.flat,
    ["episode_holdout", "episode_holdout", "spatial_holdout", "spatial_holdout"],
    ["velocity", "acceleration", "velocity", "acceleration"],
):
    candidates = [row for row in dino_motion if row["split"] == split_name and row["variable"] == variable]
    for (mode, kind), group in grouped(candidates, ["mode", "kind"]).items():
        group = sorted(group, key=lambda row: row["layer"])
        ax.plot(
            [row["layer"] for row in group], [row["r2"] for row in group],
            marker="o", label=f"{mode}: {kind}",
        )
    ax.axhline(0, color="black", lw=1)
    split_title = {
        "episode_holdout": "Unseen episodes",
        "spatial_holdout": "Unseen maze region",
    }[split_name]
    ax.set(
        title=f"{split_title}: 2-D {variable}",
        xlabel="DINO transformer block index ℓ",
        ylabel="Held-out coefficient of determination R²",
    )
    ax.legend(fontsize=7)
fig.savefig(OUTPUT_DIR / "static_vs_temporal_dino.png", dpi=180); plt.show()

control_view = sorted(
    [row for row in dino_motion if row["mode"] in ("delta", "second_delta")],
    key=lambda row: row["r2"], reverse=True,
)
show_rows(control_view, [
    "split", "variable", "representation", "r2", "position_only_r2",
    "position_residual_r2", "shuffled_q95"
], 20)

## 7. Cartesian versus polar motion

The Cartesian probe predicts $(v_x,v_y)$ or $(a_x,a_y)$ directly. The polar probes factor the same vectors into magnitude and direction: speed $\lVert v\rVert$, heading $(\cos\theta_v,\sin\theta_v)$, acceleration magnitude $\lVert a\rVert$, and acceleration direction $(\cos\theta_a,\sin\theta_a)$. Magnitude is evaluated with $R^2$; direction is evaluated with mean cosine similarity, where 1 means aligned, 0 means unrelated or orthogonal on average, and -1 means opposite. Direction metrics exclude the slowest 10% of samples because angle is unstable when magnitude is nearly zero.

In [ ]:
polar_rows = []
train_idx, test_idx = splits["episode_holdout"]
for name, rep in sorted(representations.items()):
    family, layer, kind = name.split("/", 2)
    velocity_mode = "delta" if family == "dino" else "frame"
    acceleration_mode = "second_delta" if family == "dino" else "frame"
    for variable, mode in [
        ("speed", velocity_mode), ("heading", velocity_mode),
        ("acceleration_magnitude", acceleration_mode),
        ("acceleration_direction", acceleration_mode),
    ]:
        if mode == "second_delta" and rep.shape[1] < 3:
            continue
        features, labels, _ = align_representation(rep, targets, variable, mode)
        if variable in ("heading", "acceleration_direction"):
            magnitude_variable = "speed" if variable == "heading" else "acceleration_magnitude"
            _, magnitude, _ = align_representation(rep, targets, magnitude_variable, mode)
            labels, cutoff = mask_slow_directions(labels, magnitude, train_idx, quantile=0.1)
        truth, prediction, _ = fit_probe(features, labels, train_idx, test_idx, RIDGE)
        score = direction_scores(truth, prediction) if "direction" in variable or variable == "heading" else regression_scores(truth, prediction)
        polar_rows.append({
            "representation": name, "family": family, "layer": int(layer), "kind": kind,
            "variable": variable, "mode": mode, **score,
        })

polar_metrics = polar_rows
write_rows(OUTPUT_DIR / "layerwise_polar_metrics.csv", polar_metrics)
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, (variable, score_key) in zip(axes.flat, [
    ("speed", "r2"), ("heading", "cosine"),
    ("acceleration_magnitude", "r2"), ("acceleration_direction", "cosine"),
]):
    subset = [row for row in polar_metrics if row["variable"] == variable]
    for (family, kind), group in grouped(subset, ["family", "kind"]).items():
        group = sorted(group, key=lambda row: row["layer"])
        ax.plot(
            [row["layer"] for row in group], [row[score_key] for row in group],
            marker="o", label=f"{family}: {kind}",
        )
    ax.axhline(0, color="black", lw=1)
    metric_label = (
        "Held-out coefficient of determination R²"
        if score_key == "r2" else "Mean direction cosine similarity"
    )
    ax.set(
        title=variable.replace("_", " ").title(),
        xlabel="Transformer block index ℓ", ylabel=metric_label,
    )
    ax.legend(fontsize=7)
fig.savefig(OUTPUT_DIR / "cartesian_vs_polar_by_layer.png", dpi=180)
plt.show()

## 8. Emergence table

A conservative onset is the first of two consecutive layers whose $R^2$ exceeds the shuffled-label 95th percentile and reaches at least half of that representation family's peak score. This definition should be reported alongside the full curves rather than treated as a uniquely correct boundary.

In [ ]:
onsets = []
primary = [
    row for row in metrics
    if row["split"] == "episode_holdout"
    and row["mode"] == primary_mode(row["family"], row["variable"])
]
for (family, kind, variable), group in grouped(primary, ["family", "kind", "variable"]).items():
    best = max(group, key=lambda row: row["r2"])
    onsets.append({
        "family": family, "kind": kind, "variable": variable,
        "onset_layer": readability_onset(
            group, "r2", "shuffled_q95", consecutive=2, fraction_of_peak=0.5
        ),
        "peak_r2": best["r2"], "peak_layer": int(best["layer"]),
    })
write_rows(OUTPUT_DIR / "readability_onsets.csv", onsets)
show_rows(sorted(onsets, key=lambda row: (row["variable"], row["family"], row["kind"])))

## 9. Interpretation checklist

Each point on a layer curve answers: **if the world model is frozen at this layer, how accurately can one ridge-linear map recover this physical target on examples withheld from probe training?** The plotted score evaluates the linear map, not the world model's image-prediction loss.

Read the outputs in this order:

1. Use *dataset_motion_overview.png* and *speed_by_position.png* to understand the target distribution and the risk that location predicts motion.
2. Use *readability_by_layer.png* to locate peaks and tentative emergence layers on unseen episodes.
3. Use *static_vs_temporal_dino.png* to ask whether DINO motion comes from a single image or from feature change across images.
4. Use *readability_by_layer_spatial_holdout.png* to test whether the decoder transfers to a maze region absent from probe training.
5. Use *cartesian_vs_polar_by_layer.png* to see whether vector components or magnitude/direction are more linearly organized.

Use these rules when writing the result:

1. **Position:** a high per-frame DINO score is expected and validates the probe pipeline.
2. **Velocity:** prioritize DINO $\Delta h$ and contextual predictor scores. Treat raw-frame DINO velocity as a shortcut diagnostic.
3. **Acceleration:** prioritize DINO $\Delta^2h$ and contextual predictor scores; compare its onset with velocity.
4. **Location generalization:** report whether motion performance survives the spatial holdout and beats the position-only baseline.
5. **Representation kind:** a pooled-patch result means information is distributed across the spatial token field; a CLS result means it is available from the global token; a projected-aggregate result means it survives the model's projection/aggregation stage.
6. **Causality:** never infer causal use from readability alone. Interventions or steering along probe directions are a separate experiment.

The strongest defensible claim has the form: “Using feature construction F from representation K, variable X becomes linearly readable at layer L, beats shuffled and position-only controls, remains readable after position residualization, and generalizes to a held-out maze region.”

In [ ]:
summary = {
    "config": {
        **cache_metadata, "ridge": RIDGE, "step_dt": STEP_DT,
        "episode_split": {"train_windows": len(episode_train), "test_windows": len(episode_test)},
        "spatial_split": spatial_config,
    },
    "onsets": onsets,
    "best_cartesian_rows": sorted(metrics, key=lambda row: row["r2"], reverse=True)[:20],
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("Wrote:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)